# Lab 21-09 · The RF Front End

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import frontend as fe


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}


FS = 2e9
FP = fe.fp_for_k2(FS, 0.066)
R = 1 / ((FP / FS) ** 2 - 1)
LM, CM, RM = fe.bvd(FS, 1e-12, R, 2000)


def ladder(ratio, q=2000):
    return fe.ladder(FS, FS / fe.fp_over_fs(ratio), 1e-12, 2e-12, ratio, q, 3)

## Part 1 · The resonator

*Predict* the anti-resonance for a capacitance ratio of 17.44, and where it moves if $C_0$ doubles with the motional branch unchanged.

In [2]:
print(f"C0/Cm = {R:.2f}; Cm = {CM*1e15:.2f} fF, Lm = {LM*1e9:.1f} nH, Rm = {RM:.3f} ohm")
print(f"fp = {FP/1e9:.4f} GHz; with C0 doubled fp = {FS*fe.fp_over_fs(2e-12/CM)/1e9:.4f} GHz")
print(f"|Z| at fs = {abs(fe.z_bvd(FS, 1e-12, LM, CM, RM)):.3f} ohm")
f = np.linspace(1.9e9, 2.15e9, 4001)
plt.figure()
for c0, lab_ in ((1e-12, "C0 = 1 pF"), (2e-12, "C0 = 2 pF")):
    plt.semilogy(f / 1e9, np.abs(fe.z_bvd(f, c0, LM, CM, RM)), label=lab_)
plt.xlabel("frequency (GHz)"); plt.ylabel("|Z| (ohm)"); plt.legend(); plt.show()

C0/Cm = 17.44; Cm = 57.33 fF, Lm = 110.5 nH, Rm = 0.694 ohm
fp = 2.0565 GHz; with C0 doubled fp = 2.0285 GHz
|Z| at fs = 0.694 ohm


## Part 2 · The ladder

*Predict* the 3 dB bandwidth if the coupling doubles, and the centre loss at Q = 1000.

In [3]:
for ratio, lab_ in ((R, "k2 6.6 %"), (R / 2, "coupling doubled")):
    lo, hi = fe.band_edges(ladder(ratio), FS, 0.06)
    print(f"{lab_}: 3 dB band {lo/1e6:.1f} to {hi/1e6:.1f} MHz, bandwidth {(hi-lo)/1e6:.1f} MHz")
for q in (2000, 1000, 500):
    print(f"Q = {q}: centre loss {fe.insertion_loss_db(FS, ladder(R, q))[0]:.3f} dB")
s = ngspice("ladder.cir")
print({k: round(s[k], 3) for k in ("il", "rlo", "rhi")})
f = np.linspace(1.85e9, 2.15e9, 1501)
plt.figure()
for ratio, lab_ in ((R, "k2 6.6 %"), (R / 2, "coupling doubled")):
    plt.plot(f / 1e9, -fe.insertion_loss_db(f, ladder(ratio)), label=lab_)
plt.ylim(-70, 2); plt.xlabel("frequency (GHz)"); plt.ylabel("|S21| (dB)"); plt.legend(); plt.show()

k2 6.6 %: 3 dB band 1963.9 to 2036.2 MHz, bandwidth 72.2 MHz
coupling doubled: 3 dB band 1929.8 to 2071.8 MHz, bandwidth 142.0 MHz
Q = 2000: centre loss 0.328 dB
Q = 1000: centre loss 0.655 dB
Q = 500: centre loss 1.308 dB
{'il': 0.328, 'rlo': 32.147, 'rhi': 33.197}


## Part 3 · Temperature

*Predict* the edge drift of a −20 ppm/°C filter at 1.9 GHz from −30 to +85 °C, and which way the rejection at 2.07 GHz moves as the filter warms.

In [4]:
print(f"drift {fe.tcf_drift(1.9e9, -20, 115)/1e6:.2f} MHz")
st = ladder(R)
for t in (-30, 25, 85):
    print(f"{t:4d} C: rejection at 2.07 GHz {fe.loss_at_temperature(2.07e9, st, -20, t):.2f} dB")
f = np.linspace(2.03e9, 2.10e9, 701)
plt.figure()
for t in (-30, 25, 85):
    plt.plot(f / 1e9, [-fe.loss_at_temperature(x, st, -20, t) for x in f], label=f"{t} C")
plt.axvline(2.07, ls=":", c="k"); plt.ylim(-80, 2); plt.xlabel("frequency (GHz)"); plt.ylabel("|S21| (dB)"); plt.legend(); plt.show()

drift -4.37 MHz
 -30 C: rejection at 2.07 GHz 52.98 dB
  25 C: rejection at 2.07 GHz 49.94 dB
  85 C: rejection at 2.07 GHz 47.35 dB


## Part 4 · The switch

*Predict* the isolation of a 2 Ω, 189 fF switch at 0.7, 2 and 6 GHz, and the stack for 33 dBm at VSWR 1, 2 and 4.

In [5]:
print(f"FoM {fe.fom_fs(2, 189e-15):.0f} fs, cut-off {fe.cutoff_hz(2, 189e-15)/1e9:.1f} GHz, on loss {fe.series_switch_on_db(2):.3f} dB")
for fx in (0.7e9, 2e9, 6e9):
    print(f"isolation at {fx/1e9:.1f} GHz: {fe.series_switch_off_db(fx, 189e-15):.2f} dB")
s = ngspice("switch.cir")
print({k: round(s[k], 3) for k in ("ilon", "isoff")})
for vswr in (1, 2, 4):
    print(f"VSWR {vswr}: peak {fe.peak_voltage(33, 50, vswr):.2f} V, stack {fe.stack_count(33, vswr, 2.5)}")

FoM 378 fs, cut-off 421.0 GHz, on loss 0.172 dB
isolation at 0.7 GHz: 21.64 dB
isolation at 2.0 GHz: 12.72 dB
isolation at 6.0 GHz: 4.73 dB
{'ilon': 0.172, 'isoff': 12.725}
VSWR 1: peak 14.13 V, stack 6
VSWR 2: peak 18.83 V, stack 8
VSWR 4: peak 22.60 V, stack 10


## Part 5 · Mismatch and tuning

*Predict* the VSWR at which mismatch costs 1 dB.

In [6]:
from scipy.optimize import brentq
v1 = brentq(lambda x: fe.mismatch_loss_db(x) - 1.0, 1.01, 10)
print(f"1 dB at VSWR {v1:.2f}; tuner break-even from VSWR 6 to 1.5: {fe.mismatch_loss_db(6) - fe.mismatch_loss_db(1.5):.2f} dB")
vs = np.linspace(1, 10, 200)
plt.figure(); plt.plot(vs, [fe.mismatch_loss_db(x) for x in vs])
plt.xlabel("VSWR"); plt.ylabel("mismatch loss (dB)"); plt.show()

1 dB at VSWR 2.66; tuner break-even from VSWR 6 to 1.5: 2.92 dB


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.